# 01 - Explore the IDRiD data (EDA)

**EDA** means looking at the data before training. This notebook answers: How many images do we have? What grades? Is the binary data imbalanced? Are images missing, different sizes, or exact copies? Are there near-duplicate candidates to inspect?

This baseline is a **two-class** problem: grade 0 is Normal; grades 1-4 are combined as DR. It does not predict the five original grades. Run this notebook first.

## Cell 1 - Load the small set of tools

These packages read tables, inspect images, and draw simple plots.

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imagehash

## Cell 2 - Find the project folder

If the automatic search does not find your project, set `ROOT_OVERRIDE` to the folder that contains `data` and `reports`.

In [ ]:
ROOT_OVERRIDE = None  # Example: r"C:\Users\YourName\Desktop\retinal-dr-classification"
if ROOT_OVERRIDE:
    ROOT = Path(ROOT_OVERRIDE).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    choices = [here, *here.parents, Path.home() / "Desktop" / "retinal-dr-classification"]
    ROOT = next((p for p in choices if (p / "data" / "IDRiD").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Set ROOT_OVERRIDE to the project folder.")

RUN_DIR = ROOT / "reports" / "IDRiD_clean_run"
MANIFEST = RUN_DIR / "dataset_manifest.csv"
print("Project folder:", ROOT)

## Cell 3 - Load the image list and labels

The manifest is a small table: one row per image. `grade` is the original severity grade from IDRiD.

In [ ]:
images = pd.read_csv(MANIFEST)
print("Total images:", len(images))
display(images[["image_id", "split", "grade", "relative_path"]].head())
print("Images in each source split:")
display(images.groupby("split").size().rename("images"))

## Cell 4 - Check that image files exist

This only checks paths. It does not edit or resize the original files.

In [ ]:
images["full_path"] = images["relative_path"].map(lambda name: ROOT / name)
missing = images.loc[~images["full_path"].map(Path.exists), ["image_id", "split", "relative_path"]]
print("Missing image files:", len(missing))
display(missing.head())

## Cell 5 - Count the original grades

Grade 0 means no DR. Grades 1-4 represent increasing severity.

In [ ]:
grade_counts = images.groupby(["split", "grade"]).size().unstack(fill_value=0)
display(grade_counts)
ax = grade_counts.T.plot(kind="bar", figsize=(8, 4))
ax.set_title("Original IDRiD grade counts")
ax.set_xlabel("Retinopathy grade")
ax.set_ylabel("Number of images")
plt.tight_layout()
plt.show()

## Cell 6 - Check imbalance for the two-class task

For this baseline, grade 0 becomes Normal and grades 1-4 become DR. A larger class can dominate training if we do nothing. We show the class ratio and the accuracy a trivial majority-class guess would get.

In [ ]:
images["class"] = images["grade"].map(lambda grade: "Normal" if grade == 0 else "DR")
class_counts = images.groupby(["split", "class"]).size().unstack(fill_value=0)
display(class_counts)
train_counts_before_exclusion = class_counts.loc["train"]
ratio = train_counts_before_exclusion.max() / train_counts_before_exclusion.min()
majority_accuracy = train_counts_before_exclusion.max() / train_counts_before_exclusion.sum()
print(f"Raw training imbalance: {ratio:.2f} DR/Normal ratio")
print(f"Always guessing the larger class would be correct {majority_accuracy:.1%} of the time.")

### What can we do about the imbalance?

Common choices are: (1) leave counts alone and use class-weighted loss, (2) oversample the smaller class, or (3) undersample the larger class. For this small image dataset, we chose **class-weighted loss**: keep the images as they are and give each training example a loss weight based on its class. We do not create synthetic retinal images or delete real images to force a 50/50 split. The weights are calculated later from the training partition only.

## Cell 7 - Check image dimensions

Different dimensions can reveal which dataset or camera an image came from. Here the IDRiD grading images are all expected to have the same size.

In [ ]:
image_sizes = []
for row in images.itertuples():
    with Image.open(ROOT / row.relative_path) as image:
        image_sizes.append({"image_id": row.image_id, "split": row.split,
                            "width": image.width, "height": image.height})
size_table = pd.DataFrame(image_sizes)
display(size_table.groupby(["split", "width", "height"]).size().rename("images").reset_index())

## Cell 8 - Find exact copies

SHA-256 is a file fingerprint. If two files have the same fingerprint, their bytes are exactly the same. These duplicates are confirmed; they are different from near-duplicate guesses.

In [ ]:
hash_rows = []
for row in images.itertuples():
    path = ROOT / row.relative_path
    fingerprint = hashlib.sha256(path.read_bytes()).hexdigest()
    hash_rows.append({"image_id": row.image_id, "split": row.split,
                      "grade": row.grade, "relative_path": row.relative_path,
                      "sha256": fingerprint})
hashes = pd.DataFrame(hash_rows)
hashes.to_csv(RUN_DIR / "exact_image_hashes.csv", index=False)
exact_copies = hashes[hashes.duplicated("sha256", keep=False)].sort_values("sha256")
print("Rows in exact-copy groups:", len(exact_copies))
display(exact_copies[["image_id", "split", "grade", "sha256"]])

## Cell 9 - Check exact copies across train and test

The same exact image appeared in both official splits, with grades 0 and 3. We cannot know from these files which label is correct. To avoid leakage and an unresolved target, both copies are excluded from model training and final scoring. This is recorded in a CSV; the original source images stay unchanged.

In [ ]:
split_count = hashes.groupby("sha256")["split"].nunique()
cross_split_hashes = split_count[split_count > 1].index
excluded = hashes[hashes["sha256"].isin(cross_split_hashes)].copy()
excluded["reason"] = "Exact copy crosses official train/test splits; labels conflict."
excluded.to_csv(RUN_DIR / "exact_cross_split_duplicates_excluded.csv", index=False)
display(excluded[["image_id", "split", "grade", "relative_path", "reason"]])

## Cell 10 - List near-duplicate candidates

A perceptual hash summarizes the rough visual pattern of an image. A small distance means two images look similar to this simple comparison; it does **not** prove they are the same photo. We list these for human review and do not automatically remove them.

In [ ]:
phash_rows = []
for row in images.itertuples():
    with Image.open(ROOT / row.relative_path) as image:
        fingerprint = imagehash.phash(image)
    phash_rows.append({"image_id": row.image_id, "split": row.split,
                       "grade": row.grade, "relative_path": row.relative_path,
                       "phash": fingerprint})

near_pairs = []
for left_index in range(len(phash_rows)):
    left = phash_rows[left_index]
    for right in phash_rows[left_index + 1:]:
        distance = left["phash"] - right["phash"]
        if distance <= 2:
            near_pairs.append({"image_a": left["image_id"], "split_a": left["split"],
                               "image_b": right["image_id"], "split_b": right["split"],
                               "distance": distance,
                               "path_a": left["relative_path"], "path_b": right["relative_path"]})
near_duplicates = pd.DataFrame(near_pairs)
near_duplicates.to_csv(RUN_DIR / "near_duplicate_candidates.csv", index=False)
print("Near-duplicate candidates (not confirmed duplicates):", len(near_duplicates))
display(near_duplicates.head(10))

## Cell 11 - EDA summary and next choices

Findings from this dataset: 413 official training images and 103 official test images; all 4288 x 2848; class counts are imbalanced; six exact-copy groups were found; one exact-copy pair crosses official train/test and has conflicting grades. The preprocessing notebook will exclude that pair, keep exact copies grouped during the train/validation split, and calculate class weights. Near-duplicate candidates stay in a review list unless confirmed by a person.